# The Fabry–Perot cavity against the Airy function

Companion to web-app testbench **35 “Fabry-Perot cavity”**: two R = 0.9
mirrors around 100 µm of waveguide, probed in transmission and *inside* the
cavity. The FP étalon is the drosophila of resonators — every quantity has
a closed form, so the whole testbench should land on the *Airy function*
with zero fitted parameters:

$$ T(\varphi) = \frac{T_1 T_2\, a^2}{(1 - r_1 r_2 a^2)^2}\cdot
   \frac{1}{1 + F \sin^2(\varphi/2)}, \qquad
   F = \frac{4\, r_1 r_2 a^2}{(1 - r_1 r_2 a^2)^2} $$

with $r_i = \sqrt{R_i}$, $a$ the one-way field transmission of the guide,
and $\varphi = 4\pi n(\lambda) L / \lambda$ the round-trip phase. From $F$
follow the finesse $\mathcal{F} \approx \pi\sqrt{r_1r_2a^2}/(1-r_1r_2a^2)$,
the linewidth $\mathrm{FWHM} = \mathrm{FSR}/\mathcal{F}$, and the resonant
**buildup**: the forward wave alone circulates
$B_{fwd} = T_1/(1-r_1r_2a^2)^2$ times the incident power — and the probe,
sitting at the input-mirror plane, sees the forward *and* returning waves
interfere. On resonance that plane is an antinode, so the measured power is

$$ B_{node} = B_{fwd}\,\bigl(1 + r_2\,a^2\bigr)^2 \approx 3.8\,B_{fwd} $$

— a 1 mW input piles ~36 mW onto that node. Standing waves are physics the
probe is honest about.

In [ ]:
import copy

import numpy as np
import matplotlib.pyplot as plt

from photonflux.nb import Session

C0 = 299_792_458.0
s = Session()
bench = s.load_example("35_fabry_perot_cavity")
wg = bench.instances["WG"]["settings"]
R = bench.instances["M1"]["settings"]["R"]
NEFF, NG, LAMC = wg["neff"], wg["n_group"], wg["center_wavelength_nm"]
L = wg["length_m"]
# the PAD element is a 20 dB attenuator isolating the laser: the power
# actually hitting M1 is 100 mW - 20 dB = 1 mW
pad = bench.instances["PAD"]["settings"]
p_in_mw = bench["LAS.power"] * 1e3 * 10 ** (
    -pad["loss_dB_cm"] * pad["length_m"] * 100 / 10)
print(f"L = {L * 1e6:.1f} µm, R = {R}, incident power = {p_in_mw:g} mW")


def fp(R1, R2, a2):
    """Peak transmission, F coefficient, finesse, buildup for one design."""
    q = np.sqrt(R1 * R2) * a2                # r1 r2 a² (round-trip field)
    tpk = (1 - R1) * (1 - R2) * a2 / (1 - q) ** 2
    b_fwd = (1 - R1) / (1 - q) ** 2
    return {"q": q, "tpk": tpk, "F": 4 * q / (1 - q) ** 2,
            "finesse": np.pi * np.sqrt(q) / (1 - q),
            "b_fwd": b_fwd,
            "b_node": b_fwd * (1 + np.sqrt(R2) * a2) ** 2}


a2 = 10 ** (-wg["loss_dB_cm"] * L * 100 / 10)     # one-way power transmission
d = fp(R, R, a2)
fsr = LAMC ** 2 * 1e-18 / (2 * NG * L) * 1e9      # nm — note the 2: round trip
print(f"FSR = {fsr:.4f} nm | finesse = {d['finesse']:.1f} | "
      f"FWHM = {fsr / d['finesse'] * 1e3:.1f} pm | "
      f"T_peak = {10 * np.log10(d['tpk']):.2f} dB | "
      f"buildup ×{d['b_fwd']:.1f} forward, ×{d['b_node']:.1f} at the probe")

## 1. One sweep, every FP number at once

The stored analysis sweeps 1307 → 1313 nm. We overlay the *entire* Airy
curve — dispersion included, $n(\lambda) = n_{eff} +
\frac{dn}{d\lambda}(\lambda-\lambda_c)$ with $\frac{dn}{d\lambda} =
(n_{eff}-n_g)/\lambda_c$ — and then extract FSR, linewidth, peak
transmission and buildup from the traces.

In [ ]:
res = s.run(schematic=bench)
wl = res.x
t_meas = res["trans"] / p_in_mw                   # power transmission
cav = res["cavity"] / p_in_mw                     # intracavity buildup

n_lam = NEFF + (NEFF - NG) / LAMC * (wl - LAMC)
phi = 4 * np.pi * n_lam * L * 1e9 / wl            # round-trip phase
t_airy = d["tpk"] / (1 + d["F"] * np.sin(phi / 2) ** 2)

fig, (a1, a2x) = plt.subplots(2, 1, figsize=(9.5, 5.6), sharex=True)
a1.plot(wl, 10 * np.log10(np.maximum(t_meas, 1e-9)), lw=.9,
        label="testbench 35")
a1.plot(wl, 10 * np.log10(t_airy), "--", lw=.9,
        label="Airy, zero fit parameters")
a1.set_ylabel("transmission [dB]"), a1.legend(fontsize=8)
a2x.plot(wl, cav, lw=.9, color="C2")
a2x.set_ylabel("intracavity / incident"), a2x.set_xlabel("wavelength [nm]")
a2x.set_title(f"standing-wave power at the input-mirror plane (theory ×{d['b_node']:.1f})", fontsize=9)
for a in (a1, a2x):
    a.grid(alpha=.3)
fig.tight_layout()

rms = float(np.sqrt(np.mean(
    (10 * np.log10(np.maximum(t_meas, 1e-9)) - 10 * np.log10(t_airy)) ** 2)))
print(f"RMS error over the full 6 nm sweep: {rms:.2f} dB")
assert rms < 1.0, rms

In [ ]:
def peaks(x, y, thresh, min_sep):
    idx = [i for i in range(1, len(y) - 1)
           if y[i] >= y[i - 1] and y[i] > y[i + 1] and y[i] > thresh]
    idx.sort(key=lambda i: -y[i])
    keep = []
    for i in idx:
        if all(abs(x[i] - x[j]) > min_sep for j in keep):
            keep.append(i)
    return np.array(sorted(keep), int)


def fwhm_of(x, p):
    i = int(np.argmax(p))
    h = p[i] / 2
    lo = i
    while lo > 0 and p[lo] > h:
        lo -= 1
    hi = i
    while hi < len(p) - 1 and p[hi] > h:
        hi += 1
    xl = x[lo] + (h - p[lo]) * (x[lo + 1] - x[lo]) / (p[lo + 1] - p[lo])
    xr = x[hi - 1] + (h - p[hi - 1]) * (x[hi] - x[hi - 1]) / (p[hi] - p[hi - 1])
    return xr - xl


pk = peaks(wl, t_meas, thresh=t_meas.max() / 3, min_sep=fsr / 2)
meas_fsr = float(np.diff(wl[pk]).mean())
print(f"FSR: measured {meas_fsr:.4f} nm, theory {fsr:.4f} nm")
print(f"T_peak: measured {10 * np.log10(t_meas.max()):.2f} dB, "
      f"theory {10 * np.log10(d['tpk']):.2f} dB")
print(f"buildup at probe: measured ×{cav.max():.2f}, theory "
      f"×{d['b_node']:.2f} (forward wave alone ×{d['b_fwd']:.2f})")
assert abs(meas_fsr / fsr - 1) < 0.02
assert abs(10 * np.log10(t_meas.max() / d["tpk"])) < 0.3
assert abs(cav.max() / d["b_node"] - 1) < 0.10

The λ² factor in the FSR is worth pausing on: this cavity's *round trip*
is $2L$ — the same optical length as the ring in testbench 33, which is
why the two testbenches share an FSR. And the buildup is the
other face of finesse: energy stored ∝ finesse — the same property that
made the ring's FWM conversion explode in notebook 03.

## 2. Design space: mirror reflectivity

One knob, three consequences. Sweeping $R$ on both mirrors:

* finesse sharpens as $\pi\sqrt{R a^2}/(1-Ra^2)$,
* peak transmission *falls* — with fixed waveguide loss, a higher-finesse
  cavity recirculates more times and pays the loss on every pass,
* buildup grows until loss caps it.

The high-$R$ corner is the whole tragedy of optical resonators: finesse is
free on paper and paid for in intracavity loss.

In [ ]:
ZOOM = {"mode": "dcsweep", "instance": "*", "param": "wavelength_nm",
        "start": 1309.3, "stop": 1310.7, "points": 2801}
r_sweep = [0.5, 0.7, 0.9, 0.97]
meas = {"fin": [], "tpk": [], "bld": []}
for r_i in r_sweep:
    b = copy.deepcopy(bench.doc)
    for m in ("M1", "M2"):
        b["schematic"]["instances"][m]["settings"]["R"] = r_i
    z = s.run(dict(ZOOM), schematic=b)
    tz = z["trans"] / p_in_mw
    meas["fin"].append(fsr / fwhm_of(z.x, tz))
    meas["tpk"].append(10 * np.log10(tz.max()))
    meas["bld"].append((z["cavity"] / p_in_mw).max())

th = [fp(r_i, r_i, a2) for r_i in r_sweep]
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
for ax, key, tkey, unit in [
        (axes[0], "fin", "finesse", "finesse"),
        (axes[1], "tpk", "tpk", "peak transmission [dB]"),
        (axes[2], "bld", "b_node", "peak node buildup ×")]:
    tv = [t[tkey] for t in th]
    if key == "tpk":
        tv = [10 * np.log10(v) for v in tv]
    ax.plot(r_sweep, meas[key], "o-", label="measured")
    ax.plot(r_sweep, tv, "x--", label="Airy theory")
    ax.set_xlabel("mirror R"), ax.set_ylabel(unit)
    ax.grid(alpha=.3), ax.legend(fontsize=8)
axes[0].set_yscale("log")
fig.tight_layout()

np.testing.assert_allclose(meas["fin"], [t["finesse"] for t in th], rtol=0.10)
np.testing.assert_allclose(meas["bld"], [t["b_node"] for t in th], rtol=0.10)
for m, t in zip(meas["tpk"], th):
    assert abs(m - 10 * np.log10(t["tpk"])) < 0.3

---
**Takeaways.** Six nanometres of spectrum, RMS-matched to the textbook
Airy function below a dB with nothing fitted; FSR, linewidth, peak
transmission and the ×36 standing-wave buildup all land on their closed
forms across an R = 0.5 → 0.97 design sweep.

**Things to try**

* Break the symmetry: $R_1 = 0.8, R_2 = 0.98$ halves the peak
  transmission (impedance mismatch) — the `fp()` helper above already
  takes $R_1 \ne R_2$.
* Crank `WG.loss_dB_cm` to 10 and re-run section 2: watch high-$R$
  designs collapse first.
* Testbench 36 replaces the waveguide with an SOA — gain instead of loss,
  and the same cavity *lases* (`examples/soa_fp_laser.py`).